# Node and Edge Embeddings - Using GNN

In [ ]:
import pickle
import torch
import networkx as nx
import torch.nn as nn
import matplotlib.pyplot as plt
from functions import AMLDataPreprocessing
import torch
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, auc

![Screenshot 2024-05-23 at 21.28.02.png](<attachment:Screenshot 2024-05-23 at 21.28.02.png>)

# LOADING GRAPH from Jupyter Notebook 
\(only use if not using Class Preprocessor)

In [ ]:
filename = '/Users/owhy/Documents/Datasets/HI-Small_Trans_balanced.csv'
# Create an instance of the AMLDataPreprocessing class
data_preprocessor = AMLDataPreprocessing(filename)
# Process the data
input_data, graph_full, x, y, labels, links, edges_amount, node_features, edges_features = data_preprocessor.process_data()
# Visualize
visual = data_preprocessor.visualize_graph(links, edges_amount)

adjacency_matrix = torch.from_numpy(nx.adjacency_matrix(graph_full).todense()).to(torch.float)

# PROCESS DATA directly

## Save the data

In [ ]:
with open("graph.pickle", "wb") as f:
    pickle.dump({
        'edges_features': edges_features,
        'links': links,
        'labels': labels,
        'graph_full': graph_full,
        'adjacency_matrix': adjacency_matrix,
        'visual': visual,
        'node_features': node_features,
        'x': x,
        'y': y,
        'labels': labels,
        'input_data': input_data
    }, f)


In [ ]:
input_data

In [ ]:
input_data.edge_attr.size()

In [ ]:
labels.size()

In [ ]:
input_data.x.size()

In [ ]:
node_features.size()

In [ ]:
adjacency_matrix.size()

In [ ]:
# Split the nodes into training, validation, and test sets
num_edges = edges_features.shape[0]
indices = list(range(num_edges))
print(indices)
train_indices, test_val_indices = train_test_split(indices, test_size=0.4, stratify=labels)
val_indices, test_indices = train_test_split(test_val_indices, test_size=0.5, stratify=labels[test_val_indices])

In [ ]:
# Create masks
train_mask = torch.tensor([i in train_indices for i in range(num_edges)], dtype=torch.bool)
val_mask = torch.tensor([i in val_indices for i in range(num_edges)], dtype=torch.bool)
test_mask = torch.tensor([i in test_indices for i in range(num_edges)], dtype=torch.bool)

# GRAPH NEURAL NETWORKS

In [ ]:
class GNNLayer(MessagePassing):
    def __init__(self, node_features, edge_features, hidden_channels):
        super(GNNLayer, self).__init__(aggr='add')
        self.node_features = node_features
        self.edge_features = edge_features
        self.hidden_channels = hidden_channels
        
        # Learnable parameters
        self.weight_node = nn.Parameter(torch.Tensor(node_features, hidden_channels))
        self.weight_edge = nn.Parameter(torch.Tensor(edge_features, hidden_channels))
        self.reset_parameters()
        
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_node)
        nn.init.xavier_uniform_(self.weight_edge)
        
    def forward(self, x, edge_index, edge_attr):
        # AXW0 + EW1
        global adjacency_matrix
        self.adjacency_matrix = adjacency_matrix

        # print(x.size())
        # print(self.adjacency_matrix.size())
        # print(self.weight_node.size())
        
        axw = torch.matmul(self.adjacency_matrix, x) @ self.weight_node
        ew = torch.matmul(edge_attr, self.weight_edge)

        return axw, ew

    def update(self, aggr_out):
        return aggr_out

In [ ]:
class GNNModel(nn.Module):
    def __init__(self, node_features, edge_features, hidden_channels, out_channels):
        super(GNNModel, self).__init__()
        self.conv1 = GNNLayer(node_features, edge_features, hidden_channels)

        # self.threshold = nn.Parameter(torch.tensor([0.5]))  # Trainable threshold parameter
    
    def forward(self, x, edge_index, edge_attr):
        axw1, ew1 = self.conv1(x, edge_index, edge_attr)

        head_indices, tail_indices = self.mapping(ew1, edge_index)
        scores = self.dismult(axw1, ew1, head_indices, tail_indices)
        
        return axw1, ew1, scores # returning x and e embeddings

    def update_edge_attr(self, edge_attr, new_channels):
        num_edge_features = edge_attr.size(1)
        if new_channels > num_edge_features:
            updated_edge_attr = torch.cat((edge_attr, torch.zeros((edge_attr.size(0), new_channels - num_edge_features), device=edge_attr.device)), dim=1)
        else:
            updated_edge_attr = edge_attr[:, :new_channels]
        return updated_edge_attr
    
    def dismult(self, axw, ew, head_indices, tail_indices):
        scores = []
        heads = []
        tails = []
        relations = []
        for i in range(ew.size()[0]): # going through all triples
            head = axw[head_indices[i]]
            tail = axw[tail_indices[i]]
            relation = ew[i]
            heads.append(head)
            tails.append(tail)
            relations.append(relation)
            raw_score = torch.sum(head * relation * tail, dim=-1)
            normalized_score = torch.sigmoid(raw_score)  # Apply sigmoid activation
            scores.append(raw_score) # calc scores
        scores = torch.stack(scores)
        return scores
    
    def mapping(self, ew, edge_index):
        head_indices = []
        tail_indices = []
        for c in range(ew.size()[0]): # getting all indices
            head_index = edge_index[0][c]
            tail_index = edge_index[1][c]
            head_indices.append(head_index)
            tail_indices.append(tail_index)
        
        return head_indices, tail_indices

In [ ]:
learning_rate = 0.001
hidden_channels = 25
out_channels = 15

model = GNNModel(node_features=input_data.x.size(1), edge_features=input_data.edge_attr.size(1), hidden_channels=hidden_channels, out_channels=out_channels)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()  # Binary classification loss

In [ ]:
def train(data):
    model.train()
    optimizer.zero_grad()
    x_embedding, e_embedding, scores = model(data.x, data.edge_index[:, train_mask], data.edge_attr[train_mask])

    loss = criterion(scores, labels[train_mask].float())
    loss.backward()
    optimizer.step()
    
    return loss.item(), x_embedding, e_embedding, scores

# Validation function
def validate(data):
    model.eval()
    with torch.no_grad():
        x_embedding, e_embedding, scores = model(data.x, data.edge_index[:, val_mask], data.edge_attr[val_mask])
        val_loss = criterion(scores, labels[val_mask].float()).item()
    return x_embedding, e_embedding, scores, val_loss

# Test function
def test(data):
    model.eval()
    with torch.no_grad():
        x_embedding, e_embedding, scores = model(data.x, data.edge_index[:, test_mask], data.edge_attr[test_mask])
        test_loss = criterion(scores, labels[test_mask].float()).item()
    return x_embedding, e_embedding, scores, test_loss

In [ ]:
# Continue training loop from provided script
epochs = 100
losses = []
val_losses = []
best_val_loss = float('inf')
patience = 10

all_x_embeddings = []
all_e_embeddings = []

for epoch in range(epochs):
    loss, x_embedding, e_embedding, scores = train(input_data)
    val_x_embedding, val_e_embedding, val_scores, val_loss = validate(input_data)

    losses.append(loss)
    val_losses.append(val_loss)

    all_x_embeddings.append(x_embedding.detach().cpu().numpy())
    all_e_embeddings.append(e_embedding.detach().cpu().numpy())
    # print(f"This is Fraudulent - {scores[8000]}")
    # print(f"This is Not fraudulent - {scores[2000]}")
    print(f"Epoch {epoch}, Training Loss: {loss}, Validation Loss: {val_loss}")

    # EARLY STOPPING CHECK # TODO create margin of improvement instead of just looking at if loss is improving.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save the model if validation loss improves
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter > patience:
            print(f"Validation loss hasn't improved for {patience} epochs. Early stopping...")
            break

In [ ]:
val_scores.size()

In [ ]:
# Plot
epoch_numbers = list(range(1, len(losses) + 1))

plt.figure(figsize=(10, 6))
plt.plot(epoch_numbers, losses, label="Training Loss")
plt.plot(epoch_numbers, val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.title("Training and Validation Losses Over Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# MRR --> rank, top y scores are 1s, rest is 0s --> predictions created

with torch.no_grad():
    # scores = model(heads, relations, tails).squeeze()
    sorted_indices = torch.argsort(val_scores, descending=True)
    true_indices = sorted_indices[:torch.sum(labels[val_mask].float()).int()]  # Select top predictions based on number of positive labels
    predictions = torch.zeros(len(labels[val_mask].float()))
    predictions[true_indices] = 1
    mrr = 0
    for i, idx in enumerate(true_indices):
        if labels[idx].item() == 1:
            mrr = 1 / (i + 1)  # MRR calculation
            break
    print("Mean Reciprocal Rank (MRR):", mrr)

In [ ]:
predictions.size()

In [ ]:
labels[val_mask].float().size()

In [ ]:
def evaluate_model(predictions, true_values):
    # Convert tensors to numpy arrays
    predictions = predictions.cpu().numpy()
    true_values = true_values.cpu().numpy()
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(true_values, predictions)
    precision = precision_score(true_values, predictions)
    recall = recall_score(true_values, predictions)
    f1 = f1_score(true_values, predictions)
    cm = confusion_matrix(true_values, predictions)
    classification_rep = classification_report(true_values, predictions)
    
    # ROC Curve and AUC if applicable
    try:
        fpr, tpr, thresholds = roc_curve(true_values, predictions)
        roc_auc = auc(fpr, tpr)
    except ValueError:
        fpr, tpr, roc_auc = None, None, None
    
    metrics_dict = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Confusion Matrix": cm,
        "Classification Report": classification_rep,
        "ROC Curve": (fpr, tpr, roc_auc)
    }
    
    return metrics_dict


In [ ]:
metrics_dict = evaluate_model(predictions, labels[val_mask].float())

# Print Evaluation Metrics
print("Evaluation Metrics:")
print("-------------------\n")
for metric_name, metric_value in metrics_dict.items():
    if metric_name == "Confusion Matrix":
        print("Confusion Matrix:")
        print(metric_value)
    elif metric_name == "Classification Report":
        print("Classification Report:")
        print(metric_value)
    elif metric_name == "ROC Curve":
        fpr, tpr, roc_auc = metric_value
        print("ROC Curve:")
        print("- False Positive Rate:", fpr)
        print("- True Positive Rate:", tpr)
        print("- AUC:", roc_auc)
    else:
        print(f"{metric_name}: {metric_value}")
    print()

# Save in RDF format

In [ ]:
import gzip
import torch

def save_embeddings_as_triples(edge_index, node_embeddings, edge_embeddings, file_path):
    with gzip.open(file_path, 'wt') as f:
        for i in range(edge_index.size(1)):
            head_index = edge_index[0, i].item()
            tail_index = edge_index[1, i].item()
            head_uri = f"http://example.org/node/{head_index}"
            tail_uri = f"http://example.org/node/{tail_index}"
            relation_uri = f"http://example.org/relation/{i}"

            # Write connectedTo triple
            f.write(f'<{head_uri}> <http://example.org/ontology#connectedTo> <{tail_uri}>\n')
            
            # Write head node embedding triple
            head_embedding = node_embeddings[head_index].numpy()
            head_str = ' '.join(map(str, head_embedding))
            f.write(f'<{head_uri}> <http://example.org/ontology#hasEmbedding> "{head_str}"\n')
            
            # Write tail node embedding triple
            tail_embedding = node_embeddings[tail_index].numpy()
            tail_str = ' '.join(map(str, tail_embedding))
            f.write(f'<{tail_uri}> <http://example.org/ontology#hasEmbedding> "{tail_str}"\n')
            
            # Write edge embedding triple
            relation_embedding = edge_embeddings[i].numpy()
            relation_str = ' '.join(map(str, relation_embedding))
            f.write(f'<{relation_uri}> <http://example.org/ontology#hasEmbedding> "{relation_str}"\n')

# Example usage after training
x_embeddings = torch.tensor(all_x_embeddings[-1])  # Use the last epoch's embeddings
e_embeddings = torch.tensor(all_e_embeddings[-1])  # Use the last epoch's embeddings

save_embeddings_as_triples(input_data.edge_index, x_embeddings, e_embeddings, 'embeddings.nt.gz')


In [ ]:
import gzip

def print_rdf_triples(file_path):
    # counter = 0
    with gzip.open(file_path, 'rt') as f:
        counter = 0
        for line in f:
            print(line.strip())
            counter += 1
    print(counter)

# Replace 'embeddings.nt.gz' with the actual file path
print_rdf_triples("embeddings.nt.gz") # CONNECTED TO --> HAS EMBEDDING
# print(4999*4)

In [ ]:
def evaluate_reconstruction(x_embeddings, e_embeddings, model, input_data):
    model.eval()
    reconstructed_x = model.decoder_x(torch.tensor(x_embeddings).to(input_data.x.device))
    reconstructed_e = model.decoder_e(torch.tensor(e_embeddings).to(input_data.edge_attr.device))
    
    mse_loss = nn.MSELoss()
    loss_x = mse_loss(reconstructed_x, input_data.x)
    loss_e = mse_loss(reconstructed_e, input_data.edge_attr)
    
    return loss_x.item(), loss_e.item()

loss_x, loss_e = evaluate_reconstruction(all_x_embeddings[-1], all_e_embeddings[-1], model, input_data)
print(f'Reconstruction Loss for Node Features: {loss_x}')
print(f'Reconstruction Loss for Edge Features: {loss_e}')

In [ ]:
with open("embeddings.pickle", "wb") as f:
    pickle.dump({
        'x_embeddings': x_embeddings,
        'e_embeddings': e_embeddings
    }, f)